# Predict Students' Dropout and Academic Success

### Part 1: Data Exploration

In [ ]:
# load libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from pycaret.classification import *

: 

In [ ]:
# load dataset

df = pd.read_csv("data/raw/data.csv",
    sep=";")

# Clean column names
df.columns = df.columns.str.strip()
# show first rows
df.head()

### exploratory analysis on dataset

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.info()

In [ ]:
df["Target"].value_counts()

##### Bar Chart showing the target class and how it is distributed 

In [ ]:
plt.figure(figsize=(6,4))

sns.countplot(
    data=df,
    x="Target"
)

plt.title("Student Outcome Class Distribution")
plt.xlabel("Outcome")
plt.ylabel("Number of Students")

plt.show()

In [ ]:
# count of null values in dataset
df.isnull().sum()

##### Feature Distributions 

In [ ]:
continuous_features = [
    "Previous qualification (grade)",
    "Admission grade",
    "Age at enrollment",
    "Curricular units 1st sem (grade)",
    "Curricular units 2nd sem (grade)",
]

df[continuous_features].hist(
    figsize=(12,8),
    bins=20
)

plt.tight_layout()
plt.show()

#### Tuition Fees Payed and Scholarship Based on Target

In [ ]:
plt.figure(figsize=(7,4))

sns.countplot(
    data=df,
    x="Tuition fees up to date",
    hue="Target"
)

plt.title("Tuition Payment Status vs Student Outcome")
plt.xlabel("Tuition Fees Up To Date")
plt.ylabel("Number of Students")

plt.show()

In [ ]:
plt.figure(figsize=(7,4))

sns.countplot(
    data=df,
    x="Scholarship holder",
    hue="Target"
)

plt.title("Scholarship Status vs Student Outcome")

plt.show()

##### Correlation Map

In [ ]:
# Select continuous numerical features only
continuous_features = [
    "Previous qualification (grade)",
    "Admission grade",
    "Age at enrollment",
    "Curricular units 1st sem (grade)",
    "Curricular units 2nd sem (grade)",
    "Unemployment rate",
    "Inflation rate",
    "GDP"
]

plt.figure(figsize=(10,8))

corr = df[continuous_features].corr()

sns.heatmap(
    corr,
    annot=True,
    cmap="coolwarm",
    center=0
)

plt.title("Correlation Between Continuous Features")
plt.show()

### Prep Data for Modeling

In [ ]:
# Separate features and target
X = df.drop("Target", axis=1)
y = df["Target"]

# Encode target classes
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)


In [ ]:
# Define categorical features
categorical_features = [
    "Marital status",
    "Application mode",
    "Application order",
    "Course",
    "Daytime/evening attendance",
    "Previous qualification",
    "Nacionality",
    "Mother's qualification",
    "Father's qualification",
    "Mother's occupation",
    "Father's occupation",
    "Displaced",
    "Educational special needs",
    "Debtor",
    "Tuition fees up to date",
    "Gender",
    "Scholarship holder",
    "International"
]

# Define numerical features
numerical_features = [
    "Previous qualification (grade)",
    "Admission grade",
    "Age at enrollment",
    "Curricular units 1st sem (credited)",
    "Curricular units 1st sem (enrolled)",
    "Curricular units 1st sem (evaluations)",
    "Curricular units 1st sem (approved)",
    "Curricular units 1st sem (grade)",
    "Curricular units 1st sem (without evaluations)",
    "Curricular units 2nd sem (credited)",
    "Curricular units 2nd sem (enrolled)",
    "Curricular units 2nd sem (evaluations)",
    "Curricular units 2nd sem (approved)",
    "Curricular units 2nd sem (grade)",
    "Curricular units 2nd sem (without evaluations)",
    "Unemployment rate",
    "Inflation rate",
    "GDP"
]

# Create preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ("numerical", StandardScaler(), numerical_features),
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)


In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

# Apply preprocessing
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Training shape:", X_train_processed.shape)
print("Testing shape:", X_test_processed.shape)

### Part 2: Machine Learning

##### Model 1: Logistic Regression

In [ ]:

# Create Logistic Regression model
log_reg_model = LogisticRegression(random_state=42)

# measure training time
start_train = time.perf_counter()

log_reg_model.fit(X_train, y_train)

end_train = time.perf_counter()
training_time = end_train - start_train

# measure prediction time
start_query = time.perf_counter()
predictions = log_reg_model.predict(X_test)

end_query = time.perf_counter()

query_time = end_query - start_query

# calculate accuracy
train_predictions = log_reg_model.predict(X_train)
train_accuracy = accuracy_score(y_train, train_predictions)
test_accuracy = accuracy_score(y_test, predictions)

# print results
print(f"Training Accuracy : {train_accuracy:.4f}")
print(f"Testing Accuracy  : {test_accuracy:.4f}")
print(f"Training Time     : {training_time:.6f} seconds")
print(f"Query Time        : {query_time:.6f} seconds")
print("\nClassification Report:")
print(classification_report(y_test, predictions, target_names=label_encoder.classes_))

print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, predictions)

print(pd.DataFrame(
    cm,
    index=[f"Actual {class_name}" for class_name in label_encoder.classes_],
    columns=[f"Predicted {class_name}" for class_name in label_encoder.classes_]
))



#### Model 2: Decision Tree

In [ ]:
# Create default Decision Tree model
dt_model = DecisionTreeClassifier(random_state=42)

# Train model
dt_model.fit(X_train, y_train)

# Predictions
train_predictions = dt_model.predict(X_train)
test_predictions = dt_model.predict(X_test)

# Accuracy
train_accuracy = accuracy_score(y_train, train_predictions)
test_accuracy = accuracy_score(y_test, test_predictions)

print(f"Training Accuracy: {train_accuracy:.4f}")
print(f"Testing Accuracy : {test_accuracy:.4f}")

#####  experiment with different values for max_depth

In [ ]:
depth_values = range(1, 21)

train_scores = []
test_scores = []

for depth in depth_values:
    
    model = DecisionTreeClassifier(
        max_depth=depth,
        random_state=42
    )
    
    model.fit(X_train, y_train)
    
    train_predictions = model.predict(X_train)
    test_predictions = model.predict(X_test)
    
    train_scores.append(
        accuracy_score(y_train, train_predictions)
    )
    
    test_scores.append(
        accuracy_score(y_test, test_predictions)
    )


##### model complexity curve that graphs the training and testing accuracy against the value of max_depth

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(
    depth_values,
    train_scores,
    marker="o",
    label="Training Accuracy"
)

plt.plot(
    depth_values,
    test_scores,
    marker="o",
    label="Testing Accuracy"
)

plt.xlabel("max_depth")
plt.ylabel("Accuracy")
plt.title("Decision Tree Model Complexity Curve")

plt.legend()
plt.grid(True)

plt.show()

##### find best max_depth

In [ ]:
best_depth = depth_values[test_scores.index(max(test_scores))]

print("Best max_depth:", best_depth)
print("Best Testing Accuracy:", max(test_scores))

In [ ]:
# train best max_deptj

best_dt_model = DecisionTreeClassifier(
    max_depth=best_depth,
    random_state=42
)

# Measure training time
start_train = time.perf_counter()

best_dt_model.fit(X_train, y_train)

end_train = time.perf_counter()

training_time = end_train - start_train


# Measure query time
start_query = time.perf_counter()

predictions = best_dt_model.predict(X_test)

end_query = time.perf_counter()

query_time = end_query - start_query

#  calculate final scores
train_predictions = best_dt_model.predict(X_train)

train_accuracy = accuracy_score(
    y_train,
    train_predictions
)

test_accuracy = accuracy_score(
    y_test,
    predictions
)
print(f"max_depth: {best_depth}")
print(f"Training Accuracy : {train_accuracy:.4f}")
print(f"Testing Accuracy  : {test_accuracy:.4f}")
print(f"Training Time     : {training_time:.6f} seconds")
print(f"Query Time        : {query_time:.6f} seconds")

##### Model 3: AutoML & Hyperparameter Tuning

In [ ]:
# Start timer
start_time = time.time()

# Initialize PyCaret
setup(data=df,
      target='target',
      session_id=42)

# Compare multiple models

best_model = compare_models(    
    sort='Accuracy',
    budget_time=120
)

# End timer
training_time = time.time() - start_time

print("Training/Search Time:", training_time)
print("Best Model:", best_model)

In [ ]:
# Record top 5 models
results = pull()
results.head(5)

In [ ]:

tuned_model = tune_model(best_model, optimize='Accuracy')
print(tuned_model)

In [ ]:
# training accuracy
train_results = predict_model(best_model, data=train_data)

train_accuracy = (train_results['prediction_label'] 
                  == train_results['target']).mean()

print("Training Accuracy:", train_accuracy)

In [ ]:
# testing accuracy
start_query = time.time()

test_results = predict_model(best_model, data=test_data)

query_time = time.time() - start_query

test_accuracy = (test_results['prediction_label'] 
                 == test_results['target']).mean()

print("Testing Accuracy:", test_accuracy)
print("Query Time:", query_time)

In [ ]:
plot_model(best_model, plot="feature")
plot_model(best_model, plot="confusion_matrix")
plot_model(best_model, plot="auc")